# Match Template Transpose Invariance

This notebook tests whether `skimage.feature.match_template` is transpose invariant for 3D inputs.

Fully auto-generated by Gemini.

In [1]:
import numpy as np
import skimage.feature as skf
import transpose_invariance as tpi
from skimage.util import img_as_float

In [2]:
imgs = tpi.get_3d_images()
img = img_as_float(imgs[0][:20, :64, :64])

In [3]:
# Create a template from a sub-region of the image
template = img[5:15, 20:30, 25:35].copy()

In [4]:
def func(image):
    # We must also transpose the template if we were to be fully general,
    # but for this specific test, we'll keep the template 'fixed' relative to the image content.
    # Actually, to test the function itself, if we transpose the image, we MUST transpose the template.
    # So we need a wrapper that handles both.
    pass

# Let's define a proper wrapper for match_template invariance
def match_template_wrapper(image, template):
    return skf.match_template(image, template)

def test_invariance():
    axes = (2, 1, 0)
    
    # Original match
    res_orig = skf.match_template(img, template)
    
    # Transposed match
    img_r = np.transpose(img, axes)
    template_r = np.transpose(template, axes)
    res_r = skf.match_template(img_r, template_r)
    
    # Transpose result back
    res_rolled_back = np.transpose(res_r, np.argsort(axes))
    
    diff = np.abs(res_orig - res_rolled_back).max()
    print(f"Max difference after transposition: {diff}")
    
    assert np.allclose(res_orig, res_rolled_back, atol=1e-10)

In [5]:
test_invariance()

Max difference after transposition: 2.2802593147019934e-12


## Why is it (not) invariant?

`match_template` uses `fftconvolve` and window sums. 
`fftconvolve` is generally transpose invariant (up to numerical precision).
`_window_sum_3d` is implemented using `cumsum` along each axis sequentially:

```python
def _window_sum_3d(image, window_shape):
    window_sum = _window_sum_2d(image, window_shape)
    window_sum = np.cumsum(window_sum, axis=2)
    ...
```

Since the operations are applied along each axis, and the logic is symmetric, it should be transpose invariant.